# ESSAI EN LOCAL

In [ ]:
import boto3
sts = boto3.client("sts", region_name="eu-central-1")
print(sts.get_caller_identity())


In [ ]:
# ACCES REQUIS

import boto3
from esg_lib.analysis_api import build_esg_analysis, create_analysis_boto3

def create_or_update_dashboard_from_analysis(
    aws_account_id: str,
    region: str,
    analysis_arn: str,
    dataset_arn: str,
    template_id: str,
    template_name: str,
    dashboard_id: str,
    dashboard_name: str,
):
    qs = boto3.client("quicksight", region_name=region)

    # créer ou mettre à jour le template depuis l'anlysis
    template_arn = f"arn:aws:quicksight:{region}:{aws_account_id}:template/{template_id}"

    try:
        qs.create_template(
            AwsAccountId=aws_account_id,
            TemplateId=template_id,
            Name=template_name,
            SourceEntity={
                "SourceAnalysis": {
                    "Arn": analysis_arn,
                    "DataSetReferences": [
                        {"DataSetArn": dataset_arn, "DataSetPlaceholder": "dataset"}
                    ],
                }
            },
        )
    except qs.exceptions.ResourceExistsException:
        qs.update_template(
            AwsAccountId=aws_account_id,
            TemplateId=template_id,
            Name=template_name,
            SourceEntity={
                "SourceAnalysis": {
                    "Arn": analysis_arn,
                    "DataSetReferences": [
                        {"DataSetArn": dataset_arn, "DataSetPlaceholder": "dataset"}
                    ],
                }
            },
        )

    # créer ou mettre à jour le dashboard depuis le template
    try:
        return qs.create_dashboard(
            AwsAccountId=aws_account_id,
            DashboardId=dashboard_id,
            Name=dashboard_name,
            SourceEntity={
                "SourceTemplate": {
                    "Arn": template_arn,
                    "DataSetReferences": [
                        {"DataSetArn": dataset_arn, "DataSetPlaceholder": "dataset"}
                    ],
                }
            },
        )
    except qs.exceptions.ResourceExistsException:
        return qs.update_dashboard(
            AwsAccountId=aws_account_id,
            DashboardId=dashboard_id,
            Name=dashboard_name,
            SourceEntity={
                "SourceTemplate": {
                    "Arn": template_arn,
                    "DataSetReferences": [
                        {"DataSetArn": dataset_arn, "DataSetPlaceholder": "dataset"}
                    ],
                }
            },
        )


if __name__ == "__main__":

    AWS_ACCOUNT_ID = "730335657350"
    DATASET_ARN = "arn:aws:quicksight:eu-central-1:730335657350:dataset/7ba9e6bc-aeb9-491a-b382-75909cd1ea31"
    ANALYSIS_ID = "esg-analysis-v1" 
    REGION = "eu-central-1"  

    MAPPINGS = {
        "sector": "GICS_SECTOR",
        "subsector": "NACE_GROUP_CODE",
        "country": "ISSUER_CNTRY_DOMICILE",
        "emissions_total": "COMPOSITE_INDEX",
        "company": "PORTFOLIO_NAME",
        "isin": "INSTRUMENT_ID",
        # AJOUTS pour visuals_advanced :
        "decile": "BIO_SCORE_DECILE",
        "data_quality": "DATA_QUALITY_SCORE",
    }

    #overview_sheet = build_overview_sheet(DATASET_ID, MAPPINGS)
    #overview_sheet = build_overview_sheet(DATASET_ID, MAPPINGS)
    #risk_sheet = build_risk_sheet(DATASET_ID, MAPPINGS)
    

    analysis_obj = build_esg_analysis(
        aws_account_id=AWS_ACCOUNT_ID,
        dataset_arn=DATASET_ARN,
        dataset_id="dataset",
        mappings=MAPPINGS,
        analysis_id=ANALYSIS_ID,
        analysis_name="ESG Dashboard (Code) V1",
    )

    

    # Create analysis in QuickSight
    resp_analysis = create_analysis_boto3(analysis_obj, region=REGION)
    print("Analysis:", resp_analysis)

    # Create/update dashboard from analysis
    analysis_arn = f"arn:aws:quicksight:{REGION}:{AWS_ACCOUNT_ID}:analysis/{ANALYSIS_ID}"

    resp_dash = create_or_update_dashboard_from_analysis(
        aws_account_id=AWS_ACCOUNT_ID,
        region=REGION,
        analysis_arn=analysis_arn,
        dataset_arn=DATASET_ARN,
        template_id="esg-template",
        template_name="ESG Template (Code)",
        dashboard_id="esg-dashboard",
        dashboard_name="ESG Dashboard (Code)",
    )
    print("Dashboard créé / mis à jour :", resp_dash.get("Arn", resp_dash))


In [24]:
import json
from pathlib import Path
from collections import Counter

def _extract_visual_type_and_obj(v):
    if not isinstance(v, dict):
        return type(v).__name__, {}, None

    wrapper_id = v.get("VisualId")
    visual_type = None
    for k in v.keys():
        if k != "VisualId" and k.endswith("Visual"):
            visual_type = k
            break
    if visual_type is None:
        return "Unknown", {}, wrapper_id

    return visual_type, v.get(visual_type, {}), wrapper_id


def _safe_get(d, path, default=None):
    cur = d
    for p in path:
        if not isinstance(cur, dict):
            return default
        cur = cur.get(p)
    return cur if cur is not None else default


def _collect_fields_and_aggs(obj):
    dims = set()
    measures = set()
    aggs = set()
    datasets = set()

    def walk(x):
        if isinstance(x, dict):
            if "dataSetIdentifier" in x and isinstance(x["dataSetIdentifier"], str):
                datasets.add(x["dataSetIdentifier"])

            ci = x.get("ColumnIdentifier")
            if isinstance(ci, dict):
                cn = ci.get("ColumnName")
                if isinstance(cn, str) and cn:
                    dims.add(cn)
                dsi = ci.get("DataSetIdentifier")
                if isinstance(dsi, str) and dsi:
                    datasets.add(dsi)

            col = x.get("Column")
            if isinstance(col, dict):
                cn = col.get("ColumnName")
                if isinstance(cn, str) and cn:
                    dims.add(cn)
                dsi = col.get("DataSetIdentifier")
                if isinstance(dsi, str) and dsi:
                    datasets.add(dsi)

            nmf = x.get("NumericalMeasureField")
            if isinstance(nmf, dict):
                col2 = nmf.get("Column")
                if isinstance(col2, dict):
                    cn2 = col2.get("ColumnName")
                    if isinstance(cn2, str) and cn2:
                        measures.add(cn2)
                    dsi2 = col2.get("DataSetIdentifier")
                    if isinstance(dsi2, str) and dsi2:
                        datasets.add(dsi2)

                af = nmf.get("AggregationFunction")
                if isinstance(af, dict):
                    sna = af.get("SimpleNumericalAggregation")
                    if isinstance(sna, str):
                        aggs.add(sna)

            if "SimpleNumericalAggregation" in x and isinstance(x["SimpleNumericalAggregation"], str):
                aggs.add(x["SimpleNumericalAggregation"])

            for v in x.values():
                walk(v)

        elif isinstance(x, list):
            for it in x:
                walk(it)

    walk(obj)
    dims = dims - measures
    return sorted(dims), sorted(measures), sorted(aggs), sorted(datasets)


#  Purpose par type de visual

PURPOSE_BY_TYPE = {
    "KPIVisual": "High-level KPI (valeur clé)",
    "GaugeChartVisual": "KPI vs target (jauge)",
    "BarChartVisual": "Comparaison / distribution",
    "LineChartVisual": "Évolution temporelle",
    "PieChartVisual": "Composition / parts",
    "TableVisual": "Détail des données (table)",
    "PivotTableVisual": "Détail multi-dimensions (pivot)",
    "HeatMapVisual": "Analyse croisée (heatmap)",
    "TreeMapVisual": "Répartition hiérarchique (treemap)",
    "FilledMapVisual": "Analyse géographique (carte)",
    "GeospatialMapVisual": "Analyse géographique (geo)",
    "ScatterPlotVisual": "Relation / corrélation",
    "WaterfallVisual": "Décomposition (waterfall)",
    "TextBoxVisual": "Titre / annotation",
}


def _sheet_summary_counts(sheet_visuals):
    """
    D) Résumé par sheet: KPIs / Tables / Charts / Titles
    """
    counts = {"KPIs": 0, "Tables": 0, "Charts": 0, "Titles": 0, "Other": 0}
    for v in sheet_visuals:
        vtype, _, _ = _extract_visual_type_and_obj(v)

        if vtype in ("KPIVisual", "GaugeChartVisual"):
            counts["KPIs"] += 1
        elif vtype in ("TableVisual", "PivotTableVisual"):
            counts["Tables"] += 1
        elif vtype == "TextBoxVisual":
            counts["Titles"] += 1
        elif vtype.endswith("Visual"):
            counts["Charts"] += 1
        else:
            counts["Other"] += 1
    return counts


def generate_html_dashboard_from_analysis_obj(analysis_obj, out_file="dashboard_preview.html"):
    definition = analysis_obj.get("Definition", {})
    sheets = definition.get("Sheets", [])

    # global counts
    all_types = []
    all_dims = set()
    all_meas = set()

    for sheet in sheets:
        for v in sheet.get("Visuals", []):
            vtype, vobj, _ = _extract_visual_type_and_obj(v)
            all_types.append(vtype)
            if isinstance(vobj, dict):
                dims, meas, _, _ = _collect_fields_and_aggs(vobj)
                all_dims.update(dims)
                all_meas.update(meas)

    type_counts = Counter(all_types)

    html = []
    html.append("<html><head><meta charset='utf-8'>")
    html.append("""
<style>
:root{
  --bg:#f6f8fb;
  --card:#ffffff;
  --muted:#5b667a;
  --text:#111827;
  --border:#e6ebf2;
  --shadow: 0 10px 25px rgba(17,24,39,0.08);
}
body{
  margin:0;
  font-family: ui-sans-serif, system-ui, -apple-system, Segoe UI, Roboto, Arial, sans-serif;
  background: var(--bg);
  color: var(--text);
}
.container{max-width:1180px; margin:0 auto; padding:26px 18px 60px;}
.header{
  background: var(--card);
  border:1px solid var(--border);
  box-shadow: var(--shadow);
  border-radius:16px;
  padding:18px 18px 14px;
  margin-bottom:14px;
}
.h-top{display:flex; justify-content:space-between; align-items:flex-start; gap:14px;}
h1{font-size:22px; margin:0;}
.sub{color:var(--muted); font-size:13px; margin-top:6px; line-height:1.4;}
.badges{display:flex; gap:8px; flex-wrap:wrap; justify-content:flex-end;}
.badge{
  font-size:12px;
  background:#eef2ff;
  color:#3730a3;
  border:1px solid #dbe3ff;
  padding:6px 10px;
  border-radius:999px;
  white-space:nowrap;
}
.kv{
  display:grid;
  grid-template-columns: 190px 1fr;
  gap:6px 10px;
  margin-top:10px;
  font-size:12px;
  color:var(--muted);
}
.kv b{color:#111827; font-weight:700;}
hr.sep{border:none; border-top:1px solid var(--border); margin:14px 0 10px;}
.tabs{
  display:flex;
  gap:10px;
  flex-wrap:wrap;
  padding:10px 4px 2px;
}
.tab{
  appearance:none;
  border:1px solid var(--border);
  background: var(--card);
  padding:8px 12px;
  border-radius:999px;
  cursor:pointer;
  font-size:13px;
  color:#111827;
  box-shadow: 0 6px 14px rgba(17,24,39,0.05);
}
.tab.active{
  border-color: rgba(37,99,235,0.35);
  background: rgba(37,99,235,0.08);
  color: #1d4ed8;
  font-weight:700;
}
.sheet{display:none; margin-top:10px;}
.sheet.active{display:block;}
.sheet-head{
  display:flex;
  justify-content:space-between;
  align-items:center;
  gap:12px;
  padding:6px 4px 10px;
}
.sheet-left{display:flex; flex-direction:column; gap:6px;}
h2{font-size:16px; margin:0;}
.sheet-chips{display:flex; gap:8px; flex-wrap:wrap;}
.chip{
  font-size:11px;
  background:#f3f4f6;
  border:1px solid var(--border);
  color:#111827;
  padding:4px 8px;
  border-radius:999px;
}
.grid{
  display:grid;
  grid-template-columns: repeat(12, 1fr);
  gap:12px;
}
.card{
  grid-column: span 6;
  background: var(--card);
  border:1px solid var(--border);
  border-radius:16px;
  padding:14px 14px 12px;
  box-shadow: 0 8px 18px rgba(17,24,39,0.06);
}
.card.kpi{grid-column: span 4;}
.card.full{grid-column: span 12;}
@media (max-width: 980px){
  .card, .card.kpi{grid-column: span 12;}
}
.card-top{display:flex; justify-content:space-between; align-items:flex-start; gap:10px;}
.title{font-weight:700; font-size:14px; line-height:1.25;}
.meta{display:flex; gap:8px; flex-wrap:wrap; justify-content:flex-end;}
.small{color:var(--muted); font-size:12px; margin-top:8px; line-height:1.4;}
.fields{margin-top:10px; display:flex; flex-direction:column; gap:8px;}
.row{display:flex; gap:10px; align-items:flex-start; flex-wrap:wrap;}
.label{min-width:92px; font-size:11px; color:#111827; font-weight:700;}
.pill{
  font-size:11px;
  background:#fff;
  border:1px solid var(--border);
  color:#111827;
  padding:4px 8px;
  border-radius:10px;
}
.pill.dim{background:#f0f9ff;}
.pill.meas{background:#f0fdf4;}
.pill.agg{background:#fff7ed;}
.pill.ds{background:#f5f3ff;}
.footer{
  margin-top:16px;
  color:var(--muted);
  font-size:12px;
  line-height:1.4;
  padding:0 4px;
}
</style>
""")
    html.append("</head><body><div class='container'>")

    # Header
    html.append("<div class='header'>")
    html.append("<div class='h-top'>")
    html.append("<div>")
    html.append("<h1>ESG Dashboard – Aperçu local (BI-as-Code)</h1>")
    html.append("<div class='sub'>Généré automatiquement depuis la <b>Definition</b> QuickSight. "
                "Objectif : valider la génération des sheets/visuals/fields sans dépendre de l’UI QuickSight.</div>")
    html.append("</div>")
    html.append("<div class='badges'>")
    html.append(f"<span class='badge'>Sheets: {len(sheets)}</span>")
    html.append(f"<span class='badge'>Visuals: {sum(len(s.get('Visuals', [])) for s in sheets)}</span>")
    html.append(f"<span class='badge'>Dimensions détectées: {len(all_dims)}</span>")
    html.append(f"<span class='badge'>Mesures détectées: {len(all_meas)}</span>")
    html.append("</div>")
    html.append("</div>")  # h-top

    html.append("<hr class='sep'/>")
    html.append("<div class='kv'>")
    html.append("<b>Répartition des visuels</b><span></span>")
    for t, c in type_counts.most_common():
        html.append(f"<b>{t}</b><span>{c}</span>")
    html.append("</div>")
    html.append("</div>") # header

    # Tabs
    html.append("<div class='tabs' id='tabs'>")
    for idx, sheet in enumerate(sheets):
        name = sheet.get("Name", f"Sheet {idx+1}")
        count = len(sheet.get("Visuals", []))
        safe = name.replace("<","&lt;").replace(">","&gt;")
        html.append(f"<button class='tab' data-sheet='{idx}'>🔹 {safe} <span style='opacity:.65'>({count})</span></button>")
    html.append("</div>")

    # Sheets content
    for idx, sheet in enumerate(sheets):
        sheet_name = sheet.get("Name", f"Sheet {idx+1}")
        visuals = sheet.get("Visuals", [])
        summary = _sheet_summary_counts(visuals)

        html.append(f"<div class='sheet' id='sheet_{idx}'>")

        # Sheet header summary 
        html.append("<div class='sheet-head'>")
        html.append("<div class='sheet-left'>")
        html.append(f"<h2>{sheet_name}</h2>")
        html.append("<div class='sheet-chips'>")
        html.append(f"<span class='chip'>KPIs: {summary['KPIs']}</span>")
        html.append(f"<span class='chip'>Tables: {summary['Tables']}</span>")
        html.append(f"<span class='chip'>Charts: {summary['Charts']}</span>")
        html.append(f"<span class='chip'>Titles: {summary['Titles']}</span>")
        if summary["Other"] > 0:
            html.append(f"<span class='chip'>Other: {summary['Other']}</span>")
        html.append("</div></div>")
        html.append(f"<span class='badge'>{len(visuals)} visuals</span>")
        html.append("</div>") # sheet-head

        html.append("<div class='grid'>")

        for i, v in enumerate(visuals, start=1):
            vtype, vobj, wrapper_id = _extract_visual_type_and_obj(v)

            # VisualId
            vid = wrapper_id
            if not vid and isinstance(vobj, dict):
                vid = vobj.get("VisualId")
            if not vid:
                vid = f"visual_{idx+1}_{i}"

            # Title
            title = "Sans titre"
            if vtype == "TextBoxVisual" and isinstance(vobj, dict):
                content = _safe_get(vobj, ["ChartConfiguration", "TextBoxChartConfiguration", "Content"], "")
                title = content if content else "Text box"
            else:
                t = vobj.get("Title", {}) if isinstance(vobj, dict) else {}
                if isinstance(t, dict) and t.get("Visibility") == "VISIBLE":
                    plain = _safe_get(t, ["FormatText", "PlainText"], "")
                    if plain:
                        title = plain

            dims, meas, aggs, dsets = _collect_fields_and_aggs(vobj if isinstance(vobj, dict) else {})

            # card width
            card_cls = "card"
            if vtype in ("KPIVisual", "GaugeChartVisual"):
                card_cls = "card kpi"
            if vtype in ("TableVisual", "PivotTableVisual"):
                card_cls = "card full"

            # Purpose
            purpose = PURPOSE_BY_TYPE.get(vtype, "Visualisation")

            html.append(f"<div class='{card_cls}'>")
            html.append("<div class='card-top'>")
            html.append(f"<div class='title'>{title}</div>")
            html.append("<div class='meta'>")
            html.append(f"<span class='chip'>{vtype}</span>")
            html.append(f"<span class='chip'>id: {vid}</span>")
            html.append("</div></div>")

            html.append(f"<div class='small'><b>Purpose:</b> {purpose}</div>")

            # Data lineage
            html.append("<div class='fields'>")

            if dsets:
                html.append("<div class='row'>")
                html.append("<div class='label'>Dataset</div>")
                for ds in dsets[:6]:
                    html.append(f"<span class='pill ds'>{ds}</span>")
                html.append("</div>")

            if dims:
                html.append("<div class='row'>")
                html.append("<div class='label'>Dimensions</div>")
                for d in dims[:14]:
                    html.append(f"<span class='pill dim'>{d}</span>")
                if len(dims) > 14:
                    html.append(f"<span class='pill dim'>+{len(dims)-14}</span>")
                html.append("</div>")

            if meas:
                html.append("<div class='row'>")
                html.append("<div class='label'>Mesures</div>")
                for m in meas[:14]:
                    html.append(f"<span class='pill meas'>{m}</span>")
                if len(meas) > 14:
                    html.append(f"<span class='pill meas'>+{len(meas)-14}</span>")
                html.append("</div>")

            if aggs:
                html.append("<div class='row'>")
                html.append("<div class='label'>Aggregation</div>")
                for a in aggs[:10]:
                    html.append(f"<span class='pill agg'>{a}</span>")
                html.append("</div>")

            if not dsets and not dims and not meas and not aggs:
                html.append("<div class='small'>Aucun champ détecté automatiquement.</div>")

            html.append("</div>")  # fields
            html.append("</div>")  # card

        html.append("</div></div>")  # grid + sheet

    # Footer
    html.append("<div class='footer'><b>Note :</b> Cet aperçu local valide la génération du dashboard par code (BI-as-Code). "
                "Le déploiement Template/Dashboard dans l’UI QuickSight peut être bloqué par des permissions IAM "
                "(CreateTemplate, CreateDashboard, UpdateAnalysisPermissions).</div>")


    html.append("""
<script>
(function(){
  const tabs = Array.from(document.querySelectorAll('.tab'));
  const sheets = Array.from(document.querySelectorAll('.sheet'));

  function activate(i){
    tabs.forEach(t => t.classList.remove('active'));
    sheets.forEach(s => s.classList.remove('active'));

    const tab = tabs[i];
    if(!tab) return;
    const id = tab.getAttribute('data-sheet');
    tab.classList.add('active');
    const sheet = document.getElementById('sheet_' + id);
    if(sheet) sheet.classList.add('active');
  }

  tabs.forEach((t, idx) => t.addEventListener('click', () => activate(idx)));
  activate(0);
})();
</script>
""")

    html.append("</div></body></html>")

    Path(out_file).write_text("\n".join(html), encoding="utf-8")
    return out_file


def save_analysis_json(analysis_obj, out_file="esg_analysis.json"):
    Path(out_file).write_text(json.dumps(analysis_obj, indent=2), encoding="utf-8")
    return out_file


In [25]:
save_analysis_json(analysis_obj, "esg_analysis.json")
generate_html_dashboard_from_analysis_obj(analysis_obj, "dashboard_preview.html")
print("OK -> dashboard_preview.html")

OK -> dashboard_preview.html
